# Capstone 3 - Transaction Stream Producer (Workspace A)

Instructor / model-owner notebook. Run this on a Databricks cluster in Workspace A
to emit synthetic transaction JSON events to an S3 source every 30 seconds
(~200 events/min). Workspace A's Auto Loader stream reads from that S3 path.

Output event schema (matches the proposal):

```
event_id      string    UUID
customer_id   string    acct_ universe (joins customer_credit_history)
timestamp     datetime  ISO 8601 UTC
amount        float     USD
merchant_mcc  int       4-digit MCC
channel       string    one of {pos, ecom, atm}
```

Leave this running in one notebook; build your Auto Loader ingest in another.

## Setup

Uses the standard datacouch auth cell (see `ENVIRONMENT_SETUP.md`). Writes events to
your own prefix under the course S3 bucket so streams do not collide between students.

In [ ]:
import boto3, os, json, time, uuid, random
from datetime import datetime, timezone

STUDENT_NUM = "01"  # <-- your 2-digit number

scope = f"aws-course-creds-{STUDENT_NUM}"
os.environ["AWS_ACCESS_KEY_ID"]     = dbutils.secrets.get(scope, "aws-access-key-id")
os.environ["AWS_SECRET_ACCESS_KEY"] = dbutils.secrets.get(scope, "aws-secret-access-key")
os.environ["AWS_REGION"] = os.environ["AWS_DEFAULT_REGION"] = "us-west-2"

BUCKET = "bread-academy-shared"
STREAM_PREFIX = f"capstone3/stream/student_{STUDENT_NUM}"   # your Auto Loader source path
s3 = boto3.Session(region_name="us-west-2").client("s3")
print("Writing events to s3://%s/%s/" % (BUCKET, STREAM_PREFIX))

## Event universe

`customer_id` is drawn from the same `acct_` universe as
`bread_academy.course_data.customer_credit_history`, so your point-in-time joins
line up. MCCs are the same catalog used across the course.

In [ ]:
# customer + MCC universe (consistent with customer_credit_history)
N_CUSTOMERS = 10000
MCCS = [5411, 5812, 5912, 6011, 5541, 5311, 5732, 4814, 5999, 7995, 6051, 5944, 4829, 5816, 4900]
CHANNELS = ["pos", "ecom", "atm"]

def make_event():
    cust = f"acct_{random.randint(1, N_CUSTOMERS):07d}"
    mcc = random.choice(MCCS)
    channel = "atm" if mcc == 6011 else random.choice(["pos", "pos", "ecom"])
    # amount log-normal-ish; ATM rounder
    amt = round(random.lognormvariate(3.6, 1.0) + 1.0, 2)
    if channel == "atm":
        amt = float(random.choice([20, 40, 60, 80, 100, 200]))
    return {
        "event_id": str(uuid.uuid4()),
        "customer_id": cust,
        "timestamp": datetime.now(timezone.utc).isoformat(),
        "amount": amt,
        "merchant_mcc": mcc,
        "channel": channel,
    }

print(make_event())

## Produce

Writes one small JSON-lines object per micro-batch (~100 events) every 30 seconds.
Auto Loader picks up new objects as they land. Stop the cell to stop the stream.

Tip: run for a few minutes to seed your bronze table, then keep it running while you
build the rest of the pipeline.

In [ ]:
EVENTS_PER_BATCH = 100
INTERVAL_SECONDS = 30

batch = 0
try:
    while True:
        batch += 1
        lines = "\n".join(json.dumps(make_event()) for _ in range(EVENTS_PER_BATCH))
        key = f"{STREAM_PREFIX}/events_{int(time.time())}_{batch:05d}.json"
        s3.put_object(Bucket=BUCKET, Key=key, Body=lines.encode("utf-8"))
        print(f"batch {batch}: wrote {EVENTS_PER_BATCH} events -> s3://{BUCKET}/{key}")
        time.sleep(INTERVAL_SECONDS)
except KeyboardInterrupt:
    print(f"stopped after {batch} batches ({batch * EVENTS_PER_BATCH} events)")

## Auto Loader hint (build this in your Workspace A ingest notebook)

```python
bronze = (spark.readStream.format("cloudFiles")
    .option("cloudFiles.format", "json")
    .option("cloudFiles.schemaLocation", f"s3a://{BUCKET}/capstone3/_schema/student_{STUDENT_NUM}")
    .load(f"s3a://{BUCKET}/{STREAM_PREFIX}"))

(bronze.writeStream
    .option("checkpointLocation", f"s3a://{BUCKET}/capstone3/_ckpt/student_{STUDENT_NUM}")
    .trigger(processingTime="30 seconds")
    .toTable("bread_academy.student_work.bronze_transactions_<STUDENT_NUM>"))
```
